In [3]:
import pandas as pd
import numpy as np
import s3fs

MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"
MINIO_SECRET = "adminGrupo3"
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

print("🏆 Generando las 2 tablas adicionales de alto valor analítico para la Capa Gold...")

# Cargar la tabla maestra desde Silver en memoria
df_trans = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_transaction_transformed.parquet", storage_options=storage_opts)
df_id = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_identity_transformed.parquet", storage_options=storage_opts)
df_master = pd.merge(df_trans, df_id, on="TransactionID", how="left")

# ==============================================================================
# TABLA GOLD 5: Riesgo por Dominio de Correo y Dispositivo
# ==============================================================================
gold_email_dispositivo = df_master.groupby(['P_emaildomain', 'DeviceType'], as_index=False).agg(
    total_transacciones=('TransactionID', 'count'),
    total_fraudes=('isFraud', 'sum'),
    monto_expuesto_pen=('monto_pen', 'sum')
)

gold_email_dispositivo['tasa_fraude_pct'] = (
    gold_email_dispositivo['total_fraudes'] / gold_email_dispositivo['total_transacciones'] * 100
).round(2)

gold_email_dispositivo = gold_email_dispositivo[gold_email_dispositivo['total_transacciones'] > 100]
gold_email_dispositivo.to_parquet(
    f"s3://{BUCKET_NAME}/gold/kpis_riesgo_email_dispositivo.parquet", 
    index=False, 
    storage_options=storage_opts
)

# ==============================================================================
# TABLA GOLD 6: Comportamiento por Frecuencia de Tarjeta (card1)
# ==============================================================================
df_card_freq = df_master.groupby('card1', as_index=False).agg(
    transacciones_por_tarjeta=('TransactionID', 'count'),
    fraudes_por_tarjeta=('isFraud', 'sum'),
    monto_total_tarjeta=('monto_pen', 'sum')
)

df_card_freq['rango_frecuencia_tarjeta'] = pd.cut(
    df_card_freq['transacciones_por_tarjeta'],
    bins=[0, 1, 5, 20, np.inf],
    labels=['1_Única (1 tx)', '2_Baja (2-5 tx)', '3_Media (6-20 tx)', '4_Alta (>20 tx)']
).astype(str)

gold_frecuencia_tarjetas = df_card_freq.groupby('rango_frecuencia_tarjeta', as_index=False).agg(
    total_tarjetas=('card1', 'count'),
    total_transacciones=('transacciones_por_tarjeta', 'sum'),
    fraudes_totales=('fraudes_por_tarjeta', 'sum'),
    monto_total_pen=('monto_total_tarjeta', 'sum')
)

gold_frecuencia_tarjetas['tasa_fraude_pct'] = (
    gold_frecuencia_tarjetas['fraudes_totales'] / gold_frecuencia_tarjetas['total_transacciones'] * 100
).round(2)

gold_frecuencia_tarjetas.to_parquet(
    f"s3://{BUCKET_NAME}/gold/kpis_frecuencia_tarjetas.parquet", 
    index=False, 
    storage_options=storage_opts
)

print("✨ ¡Tablas de dominios de correo y frecuencia de tarjetas añadidas a la Capa Gold con éxito!")

print("\n📊 Vista previa - Riesgo por Correo y Dispositivo:")
display(gold_email_dispositivo.sort_values(by='tasa_fraude_pct', ascending=False).head(5))

print("\n📊 Vista previa - Comportamiento por Frecuencia de Tarjeta:")
display(gold_frecuencia_tarjetas.sort_values(by='tasa_fraude_pct', ascending=False))

🏆 Generando las 2 tablas adicionales de alto valor analítico para la Capa Gold...
✨ ¡Tablas de dominios de correo y frecuencia de tarjetas añadidas a la Capa Gold con éxito!

📊 Vista previa - Riesgo por Correo y Dispositivo:


,P_emaildomain,DeviceType,total_transacciones,total_fraudes,monto_expuesto_pen,tasa_fraude_pct
102,outlook.com,desktop,1298,260,2.815193e+05,20.03
105,outlook.es,desktop,178,34,2.977647e+04,19.10
103,outlook.com,mobile,1018,159,1.850000e+05,15.62
49,gmail.com,mobile,24038,3304,6.354646e+06,13.74
33,desconocido,mobile,2819,367,4.241544e+05,13.02



📊 Vista previa - Comportamiento por Frecuencia de Tarjeta:


,rango_frecuencia_tarjeta,total_tarjetas,total_transacciones,fraudes_totales,monto_total_pen,tasa_fraude_pct
0,1_Única (1 tx),3444,3444,146,2.045304e+06,4.24
3,4_Alta (>20 tx),2535,537795,19210,2.694735e+08,3.57
1,2_Baja (2-5 tx),4219,13014,381,7.432266e+06,2.93
2,3_Media (6-20 tx),3355,36287,926,2.006964e+07,2.55
